# Offline Diffusion DQN

Copy of example 2 with a diffusion head conditioned on the backbone's final
readout features. The environment, tokenizer, encoder, and data pipeline are unchanged.
We extend the Q-only flow-matching loss of
[Value Forcing, Appendix D.5](https://arxiv.org/html/2611.02160v1#A4.SS5).
The action values evolve jointly from independent Gaussian coordinates.
One Bellman backup reaches the next decision; every diffusion integration step
contributes a velocity loss. There are no n-step or multi-step horizon options.


In [ ]:
import json
from copy import deepcopy

import torch
from torch import nn

from mouse_core import AdamW, polyak_update
from mouse_core.data import (
    Augmenter,
    DataLoader,
    Tokenizer,
    compose,
    load_stores_from_hub,
    save_tokenizer,
)
from mouse_core.models import (
    Backbone,
    DecodeCache,
    Embedding,
    Model,
    MultiEmbedding,
    prepare_decode,
    save_model,
)
from mouse_core.objectives import DiffusionDqnObjective, boundary_discount

# Repositories
# Hugging Face dataset repo for load_stores_from_hub
DATASET_ID = "mouse-example-frozenlake-dataset"
# Local checkpoint directory
OUTPUT = "runs/offline-diffusion-dqn"

# Environment
# number of discrete actions predicted by the head
MAX_ACTIONS = 4
# vocabulary size for discrete observations
MAX_OBS_DISCRETE = 64
# max steps per episode (matches collect)
MAX_STEPS_PER_EPISODE = 24
# inner episodes per outer task (matches collect)
REPEAT_COUNT = 20

# Model
# HF checkpoint for Tokenizer, Embedding, and Backbone
PRETRAINED = "Qwen/Qwen3-0.6B"
DIFFUSION_HIDDEN_DIM = 128
DIFFUSION_STEPS = 10
# Independent Monte Carlo samples for greedy action scores
RETURN_SAMPLES = 8
# final prediction linear's initialization
HEAD_SCALE = 0.1

# Compute text before each value readout
COMPUTE_TOKENS = 8

# Training
# packed tokens; whole episodes are added until the next one would not fit
TOKEN_BUDGET = 16384
# outer train cycles (print cadence)
NUM_CYCLES = 2
# optimizer updates per cycle (passed to run_train)
TRAIN_STEPS = 50

# Optimizer learning rates
# token embeddings and numeric Fourier features
LR_EMBEDDING = 1e-05
# backbone
LR_BACKBONE = 1e-05
# prediction heads and their input RMSNorms
LR_HEADS = 1e-05

# Target interpolation rates
# delayed embedding
POLYAK_TAU_EMBEDDING = 0.1
# delayed backbone
POLYAK_TAU_BACKBONE = 0.1
# delayed value head
POLYAK_TAU_VALUE = 0.001

# Runtime
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

## Load Data

`load_stores_from_hub` downloads the dataset snapshot and reconstructs the saved
`Datastore` objects. Each returned store is one ordered environment stream.


In [ ]:
stores = load_stores_from_hub(repo_id=DATASET_ID, split="train", force_download=True)

## Data Pipeline

`DataLoader` samples complete outer tasks with
`sample_ranges=sample_ranges`, keeping all repeated lake episodes together.
Each sampled task gets a unique sequence ID so attention and objective transitions
cannot cross between tasks.

Pass exactly one of `token_budget` or `events_budget`. Each batch contains whole samples
up to that budget. An oversized first sample raises. Token budgets count packed tokens,
including episode prompts; event budgets count stored source events, not decision
readouts.

The pipeline is
`source events → sparse decisions → augmenter → tokenizer → pack → embedding`:

| Stage       | Role                                                                                                             |
| ----------- | ---------------------------------------------------------------------------------------------------------------- |
| `Augmenter` | Permutes actions and observations consistently within a sampled task; action permutations also update `_q_star`. |
| `Tokenizer` | Tokenizes matching fields per event, groups tokens by `step_index`, and retains explicit objective labels.       |

`train_transform = compose(stages=(augmenter.events, tokenizer.encode))` is passed as
the loader’s `transform`. `next_batch()` returns `(inputs, objective_data)` as
TensorDicts; pass `objective_data["sequence_ids"]` as the objective’s `sequence_ids`.
[Inference](15_inference.ipynb) uses the saved tokenizer without augmentation so
selected actions match the environment.

For per-event encoding, configure `Tokenizer` with a flat `input_fields` list,
`step_field="step_index"`, and `episode_field="episode_index"`, then pass
`transform=tokenizer.encode` directly to the loader. It accumulates event tokens until
the step marker changes. Define a local range function to select windows of whole steps
N whole steps within an episode. Augmentation for this path must transform event
payloads before encoding; `augmenter.events` preserves source events and shares
episode-keyed draws.

`COMPUTE_TOKENS` sets a constant number of compute tokens before every value readout.
Set it to zero to disable compute text. The field filters on `action_requested`
and uses the literal format `" ." * COMPUTE_TOKENS`. No callback or runtime context
is needed. The configured text is saved with the tokenizer and reused at inference.

Qwen encodes each spaced period as one token (ID 659); ten bare periods can merge
into just one token. Passive and
terminal events emit no compute text. Compute tokens count toward the batch budget.

Only the newline has `head_output=True`. The diffusion loss trains the compute
positions through causal attention.


In [ ]:
def action_name(action):
    return ("a", "b", "c", "d")[action]


episode_prompt = (
    "Your job is to predict the future sum of rewards in FrozenLake. "
    "Navigate a grid; reach the goal for reward; a hole ends the "
    "episode with none. Each episode lasts at most "
    f"{MAX_STEPS_PER_EPISODE} steps. You have {REPEAT_COUNT} "
    "episodes to solve the task. The grid is permuted, so squares are "
    "not in order; action ids may be remapped, then rendered as a, b, c, d.\n"
    "Strategy: explore; keep a mental map of what has and has not been "
    "explored; avoid holes you have already fallen in; once you have a "
    "path to the goal, repeat it.\n"
    "At each action-request newline, predict the value of all actions. Format:\n"
    "action letter,observation[,r=reward][,terminated][,truncated].\nx marks no action at the beginning of an episode.\n"
)

augmenter = Augmenter(
    seed_field="episode_index",
    fields=[
        {
            "type": "discrete_permute",
            "provider": None,
            "input_field": "action",
            "input_vector_field": "_q_star",
            "vocab_size": MAX_ACTIONS,
        },
        {
            "type": "discrete_permute",
            "provider": None,
            "input_field": "observation",
            "vocab_size": MAX_OBS_DISCRETE,
        },
    ],
)

tokenizer = Tokenizer(
    tokenizers={"text": PRETRAINED},
    step_field="step_index",
    input_fields=[
        {
            "type": "text",
            "tokenizer": "text",
            "provider": "coordinator",
            "input_field": "_episode_start",
            "format": episode_prompt,
        },
        {
            "type": "text",
            "tokenizer": "text",
            "provider": "environment",
            "input_field": "repeat_episode_index",
            "format": 'episode {repeat_episode_index+1}\n',
        },
        {
            "type": "text",
            "tokenizer": "text",
            "provider": "environment",
            "input_field": "repeat_episode_index",
            "format": "x",
        },
        {
            "type": "text",
            "tokenizer": "text",
            "provider": "agent",
            "input_field": "action",
            "format": '{action_name(action)}',
            "format_functions": {"action_name": action_name},
        },
        {
            "type": "text",
            "tokenizer": "text",
            "provider": "environment",
            "input_field": "observation",
            "format": ',{observation}',
        },
        {
            "type": "text",
            "tokenizer": "text",
            "provider": "environment",
            "input_field": "reward",
            "format": ',r={reward:g}',
        },
        {
            "type": "text",
            "tokenizer": "text",
            "provider": "environment",
            "input_field": "inner_terminated",
            "format": ",terminated",
        },
        {
            "type": "text",
            "tokenizer": "text",
            "provider": "environment",
            "input_field": "inner_truncated",
            "format": ",truncated",
        },
        {
            "type": "text",
            "tokenizer": "text",
            "provider": "environment",
            "input_field": "action_requested",
            "format": " ." * COMPUTE_TOKENS,
        },
        {
            "type": "token",
            "tokenizer": "text",
            "provider": "environment",
            "input_field": "action_requested",
            "format": "\n",
            "head_output": True,
        },
        {
            "type": "text",
            "tokenizer": "text",
            "provider": "environment",
            "input_field": ["inner_terminated", "inner_truncated"],
            "format": "\n\n",
        },
    ],
    objective_fields=[
        {
            "input_field": "action",
            "missing_value": None,
        },
        {
            "input_field": "reward",
            "missing_value": 0.0,
        },
        {
            "input_field": "_terminated",
            "output_field": "terminated",
            "missing_value": False,
        },
        {
            "input_field": "_truncated",
            "output_field": "truncated",
            "missing_value": False,
        },
    ],
)


train_transform = compose(stages=(augmenter.events, tokenizer.encode))


def sample_ranges(dataset):
    """Keep all events belonging to one outer episode together."""
    start, previous = 0, None
    for index, episode in enumerate(dataset["episode_index"][:]):
        if episode is not None and episode != previous:
            if index:
                yield start, index
            start, previous = index, episode
    if len(dataset):
        yield start, len(dataset)


def sample_eligible(events):
    """Train only on episodes with a recorded final environment result."""
    last = events[-1]
    return last["provider"] == "environment" and (
        last["payload"].get("_terminated") is True
        or last["payload"].get("_truncated") is True
    )


loader = DataLoader(
    stores=stores,
    token_budget=TOKEN_BUDGET,
    sample_ranges=sample_ranges,
    sample_eligible=sample_eligible,
    transform=train_transform,
    prefetch=4,
    num_workers=4,
)

## Build Encoder and Diffusion Head

The encoder is copied verbatim from example 2. `DiffusionHead` reads its final
features, the complete noisy action-value vector, and diffusion time. A dense
MLP outputs all action velocities together, so each velocity can depend on every
action's current value. Euler integration runs only in the head.

Each trajectory starts from independent standard Gaussian coordinates `[P,A]`.
The delayed encoder and head are frozen Polyak copies. Greedy actions use mean
integrated returns from separate noise draws; a separate delayed trajectory
supplies the training paths. Only online parameters are optimized.


In [ ]:
class Encoder(Model):
    def __init__(
        self, *, pretrained, tokenizer_sources, dtype
    ):
        super().__init__()
        self.embedding = MultiEmbedding(
            embeddings={
                name: Embedding(pretrained=source)
                for name, source in tokenizer_sources.items()
            }
        )
        self.backbone = Backbone(
            pretrained=pretrained,
            dtype=dtype,
        )

    def forward(self, tokens, *, cache=None, use_cache=False):
        if cache is not None and not use_cache:
            raise ValueError("Passing cache= requires use_cache=True.")
        tensors = tokens.to(next(self.parameters()).device)
        token_ids = tensors["token_ids"]
        numeric_values = tensors["numeric_values"]
        numeric_mask = tensors["numeric_mask"]
        sequence_ids = tensors["sequence_ids"]
        head_token_ids = tensors["head_mask"].nonzero(as_tuple=True)[0]
        embed = self.embedding(
            token_ids,
            numeric_values=numeric_values,
            numeric_mask=numeric_mask,
            numeric_field_ids=tensors["numeric_field_ids"],
            numeric_field_names=tokens["numeric_field_names"],
            tokenizer_ids=tensors["tokenizer_ids"],
            tokenizer_names=tokens["tokenizer_names"],
        )
        if use_cache:
            embed, lengths, head_token_ids, _ = prepare_decode(
                embeds=embed,
                tokens=tokens,
                sequence_ids=sequence_ids,
                head_mask=tensors["head_mask"],
            )
            if cache is None:
                cache = DecodeCache(
                    sessions=(
                        self.backbone.decode_session(
                            batch_size=tokens["num_sequences"]
                        ),
                    )
                )
            if len(cache.sessions) != 1:
                raise ValueError("expected one backbone cache session")
            features = self.backbone(embed, cache=cache.sessions[0], lengths=lengths)
            head_gather_ids = head_token_ids.unsqueeze(-1).expand(
                *head_token_ids.shape,
                features.shape[-1],
            )
            h = features.gather(1, head_gather_ids)
        else:
            features = self.backbone(embed, sequence_ids=sequence_ids)
            h = features[head_token_ids]
        return {
            "z": h,
            "cache": cache,
        }


class DiffusionHead(nn.Module):
    times: torch.Tensor

    def __init__(self, *, hidden_dim, num_actions, width, num_steps, scale):
        super().__init__()
        if num_steps < 1:
            raise ValueError("num_steps must be positive")
        self.num_actions = num_actions
        self.width = width
        self.num_steps = num_steps
        self.norm = nn.RMSNorm(hidden_dim)
        self.context = nn.Linear(hidden_dim, width)
        self.noisy_return_time = nn.Linear(num_actions + 1, width)
        self.output = nn.Linear(width, num_actions)
        with torch.no_grad():
            self.output.weight.mul_(scale)
            self.output.bias.mul_(scale)
        self.register_buffer("times", torch.arange(num_steps).float() / num_steps)

    def forward(self, z, samples, *, times):
        """Final features [P,H], noisy returns [P,T,A], times [T] -> velocities."""
        context = self.context(self.norm(z.float()))
        time = times[None, :, None].expand(*samples.shape[:-1], 1)
        noisy = self.noisy_return_time(torch.cat([samples, time], dim=-1))
        return self.output(torch.nn.functional.silu(context[:, None] + noisy))

    @torch.no_grad()
    def trajectory(self, z, *, initial_noise):
        """Evolve the full action vector; initial_noise has shape [P,A]."""
        if initial_noise.shape != (z.shape[0], self.num_actions):
            raise ValueError("initial_noise must have shape [P, A]")
        x = initial_noise
        samples, velocities = [], []
        for k in range(self.num_steps):
            v = self(z, x[:, None, :], times=self.times[k:k + 1])[:, 0]
            samples.append(x)
            velocities.append(v)
            x = x + v / self.num_steps
        return torch.stack(samples, 1), torch.stack(velocities, 1), x

    @torch.no_grad()
    def q_values(self, z, *, num_samples):
        """Average joint return vectors initialized with independent action noise."""
        leading_shape = z.shape[:-1]
        z = z.reshape(-1, z.shape[-1])
        features = z[:, None, :].expand(-1, num_samples, -1).reshape(-1, z.shape[-1])
        noise = torch.randn(features.shape[0], self.num_actions, device=z.device)
        _, _, returns = self.trajectory(features, initial_noise=noise)
        values = returns.reshape(z.shape[0], num_samples, self.num_actions).mean(1)
        return values.reshape(*leading_shape, self.num_actions)


encoder = (
    Encoder(
        tokenizer_sources=tokenizer.tokenizer_sources,
        pretrained=PRETRAINED,
        dtype=torch.float32,
    )
    .train()
    .to(device)
)

q_head = (
    DiffusionHead(
        hidden_dim=encoder.backbone.hidden_dim,
        num_actions=MAX_ACTIONS,
        width=DIFFUSION_HIDDEN_DIM,
        num_steps=DIFFUSION_STEPS,
        scale=HEAD_SCALE,
    )
    .train()
    .to(device)
)

# A checkpoint container; forward calls use the encoder and heads separately.
modules = nn.ModuleDict({"encoder": encoder, "q_head": q_head})

delayed_encoder = deepcopy(encoder).requires_grad_(False).eval()
backup_value_head = deepcopy(q_head).requires_grad_(False).eval()

print(encoder)
print(q_head)

## Train

For the action taken, at each diffusion time `t`, construct the Bellman input
and velocity target:

- `x = t*r + g*z_next + (1-t)*sqrt(1-g*g)*noise`
- `target = r + g*v_next - sqrt(1-g*g)*noise`

Here `g` is the supplied discount, including termination. At outer termination
it is zero, so `x = t*r + (1-t)*noise` and `target = r-noise`; no terminal
readout is needed. Rewards keep example 2's scale. The optional `(1-gamma)`
reward normalization in the paper is not applied.

As in example 2, inner episode endings keep the return connected across repeats.
The existing one-step DQN alignment carries automatic reset rows to the next
decision, combining their rewards and discounts. The correction uses this combined
discount. It never continues through another executed decision or sample boundary.

Start the online training input from the delayed **current-state** action vector,
then replace only the taken action's coordinate with `x`. Its target velocity is
`target`. The head reads the entire vector and predicts all action velocities,
but the loss selects only the action taken. Other output coordinates receive no
direct loss; shared head parameters still train through the selected output.

This is an experimental joint-head extension of the paper's scalar Q-only loss.
The paper's flow-matching argument requires the conditional velocity of a specified
path. Here each taken action defines a different vector input distribution and
supervises only one velocity coordinate. Matching these partial targets does not
establish transport of any one joint path, or Bellman-consistent generated
marginals. Replay also does not identify the true joint distribution of returns
for actions that were not taken. These are limits of the derivation, not evidence
that the coupled head cannot work.

The delayed path and correction noise are independent. Action scores use separate
Monte Carlo draws, so the greedy choice is fixed for the entire training path.
`prepare` builds detached inputs and targets; the online head then predicts
velocities. Loss averages only taken-action errors over valid readouts and
**all** `DIFFUSION_STEPS`. During deployment, integrate the whole action vector
and average return samples separately for each action before taking `argmax`.


In [ ]:
optimizer = AdamW(
    params=[
        *encoder.parameter_groups(
            lr={"embedding": LR_EMBEDDING, "backbone": LR_BACKBONE}
        ),
        {"params": q_head.parameters(), "lr": LR_HEADS},
    ],
    weight_decay=0.0,
    betas=(0.9, 0.95),
    eps=1e-08,
)
objective = DiffusionDqnObjective(
    reward=None,
    discount=boundary_discount(
        gamma_step=1.0,
        gamma_terminated=0.0,
        gamma_truncated=0.0,
    ),
)


def run_train(
    *, num_steps: int
) -> tuple[torch.Tensor, dict[str, float | torch.Tensor]]:
    """Run ``num_steps`` optimizer steps on batches from ``loader``."""
    modules.train()
    loss: torch.Tensor | None = None
    metrics: dict[str, float | torch.Tensor] = {}
    for _ in range(num_steps):
        inputs, objective_data = loader.next_batch()
        inputs = inputs.to(device)
        objective_data = objective_data.to(device)
        sequence_ids = objective_data["sequence_ids"]
        out = encoder(inputs)
        with torch.no_grad():
            delayed_out = delayed_encoder(inputs)
            features = delayed_out["z"]
            action_values = backup_value_head.q_values(features, num_samples=RETURN_SAMPLES)
            initial_noise = torch.randn(features.shape[0], q_head.num_actions, device=device)
            samples, velocities, _ = backup_value_head.trajectory(
                features, initial_noise=initial_noise,
            )
            flow_inputs, targets, actions, weight = objective.prepare(
                objective_data=objective_data,
                sequence_ids=sequence_ids,
                step_ids=objective_data["step_ids"],
                backup_samples=samples,
                backup_value_predictions=velocities,
                backup_action_predictions=action_values,
                times=q_head.times,
                noise=torch.randn(features.shape[0], device=device),
            )
        predictions = q_head(out["z"], flow_inputs, times=q_head.times)
        losses, metrics = objective(
            predictions=predictions, targets=targets, actions=actions, weight=weight,
        )
        loss = torch.stack(tuple(losses.values())).sum()
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        polyak_update(
            target=delayed_encoder.embedding,
            source=encoder.embedding,
            tau=POLYAK_TAU_EMBEDDING,
        )
        polyak_update(
            target=delayed_encoder.backbone,
            source=encoder.backbone,
            tau=POLYAK_TAU_BACKBONE,
        )
        polyak_update(target=backup_value_head, source=q_head, tau=POLYAK_TAU_VALUE)
    assert loss is not None
    return (loss, metrics)

## Run

Each cycle calls `run_train(num_steps=TRAIN_STEPS)`. For inference, pass the
encoder's final features to `q_head.q_values(z, num_samples=RETURN_SAMPLES)`
and choose the highest-scoring action. Notebook 15's scalar head must be replaced
with this notebook's `DiffusionHead` when loading this checkpoint.


In [ ]:
for cycle in range(NUM_CYCLES):
    loss, metrics = run_train(num_steps=TRAIN_STEPS)
    print(
        f"cycle={cycle} train  loss={loss.item():.4f}  "
        f"diffusion_steps={torch.as_tensor(metrics['per_diffusion_step']).tolist()}"
    )

loader.close()

## Save Locally

Save the encoder, diffusion head, and unchanged tokenizer. Reconstruct the same
components and settings before loading. Publishing is a separate reviewed step.


In [ ]:
modules.eval().to("cpu")
save_model(model=modules, path=OUTPUT)
save_tokenizer(tokenizer=tokenizer, path=OUTPUT)
print(f"Saved encoder, diffusion head, and tokenizer to {OUTPUT}")
